In [28]:
import pandas as pd

# Load the raw data file
file_path = "/home/kaifalam/vishvajeet_verma/modaxe_obd_logger/data/2025-10-25 11-06-00.csv"
df = pd.read_csv(file_path, sep=';', engine='python')

# Display structure and a preview
print("Columns in the file:")
print(df.columns)

print("\nShape of the data (rows, columns):", df.shape)

print("\nFirst 50 rows:")
display(df.head(50))


Columns in the file:
Index(['SECONDS', 'PID', 'VALUE', 'UNITS', 'LATITUDE', 'LONGTITUDE',
       'Unnamed: 6'],
      dtype='object')

Shape of the data (rows, columns): (229877, 7)

First 50 rows:


,SECONDS,PID,VALUE,UNITS,LATITUDE,LONGTITUDE,Unnamed: 6
0,39973.005601,Average speed (GPS),5.356695,km/h,13.016818,77.562598,NaN
1,39973.005601,Speed (GPS),5.657209,km/h,13.016818,77.562598,NaN
2,39973.011601,Altitude (GPS),812.500000,m,13.016818,77.562598,NaN
3,39973.015601,Altitude (GPS),812.500000,m,13.016818,77.562598,NaN
4,39973.015601,Average speed (GPS),5.357241,km/h,13.016818,77.562598,NaN
5,39973.015601,Speed (GPS),5.657209,km/h,13.016818,77.562598,NaN
6,39973.017601,Altitude (GPS),866.599976,m,13.016818,77.562598,NaN
7,39973.017601,Altitude (GPS),812.500000,m,13.016818,77.562598,NaN
8,39973.017601,Average speed (GPS),5.357397,km/h,13.016818,77.562598,NaN
9,39973.017601,Average speed (GPS),5.357399,km/h,13.016818,77.562598,NaN


In [29]:
print("Exact column names:\n")
for i, col in enumerate(df.columns):
    print(f"{i+1}. '{col}'")


Exact column names:

1. 'SECONDS'
2. 'PID'
3. 'VALUE'
4. 'UNITS'
5. 'LATITUDE'
6. 'LONGTITUDE'
7. 'Unnamed: 6'


In [31]:
import pandas as pd
from datetime import datetime, timedelta

# 1️⃣ Drop unwanted unnamed column
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# 2️⃣ Define your actual recording start date
base_date = datetime(2025, 10, 17, 0, 0, 0)  # 17 Oct 2025

# 3️⃣ Convert 'SECONDS' → timestamps relative to that base date
start_time = df['SECONDS'].iloc[0]
df['timestamp'] = base_date + pd.to_timedelta(df['SECONDS'] - start_time, unit='s')

# 4️⃣ Build expanded DataFrame (no collapsing)
rows = []
for _, row in df.iterrows():
    new_row = {
        'timestamp': row['timestamp'],
        row['PID']: row['VALUE'],
        'LATITUDE': row['LATITUDE'],
        'LONGITUDE': row['LONGTITUDE']
    }
    rows.append(new_row)

pivot_df = pd.DataFrame(rows)

# 5️⃣ Ensure all PIDs are present
all_pids = sorted(df['PID'].unique().tolist())
pivot_df = pivot_df.reindex(columns=['timestamp'] + all_pids + ['LATITUDE', 'LONGITUDE'])

# 6️⃣ Display result
print("✅ Data transformed successfully with correct timestamps (2025).")
print("Shape of transformed data:", pivot_df.shape)
display(pivot_df.head(10))


✅ Data transformed successfully with correct timestamps (2025).
Shape of transformed data: (229877, 34)


,timestamp,Altitude (GPS),Average fuel consumption,Average fuel consumption (total),Average fuel consumption 10 sec,Average speed,Average speed (GPS),Barometric pressure,Calculated boost,Calculated engine load value,...,Intake air temperature,Intake manifold absolute pressure,MAF air flow rate,OBD Module Voltage,Power from MAF,Speed (GPS),Vehicle acceleration,Vehicle speed,LATITUDE,LONGITUDE
0,2025-10-17 00:00:00.000,NaN,NaN,NaN,NaN,NaN,5.356695,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.016818,77.562598
1,2025-10-17 00:00:00.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,5.657209,NaN,NaN,13.016818,77.562598
2,2025-10-17 00:00:00.006,812.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.016818,77.562598
3,2025-10-17 00:00:00.010,812.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.016818,77.562598
4,2025-10-17 00:00:00.010,NaN,NaN,NaN,NaN,NaN,5.357241,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.016818,77.562598
5,2025-10-17 00:00:00.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,5.657209,NaN,NaN,13.016818,77.562598
6,2025-10-17 00:00:00.012,866.599976,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.016818,77.562598
7,2025-10-17 00:00:00.012,812.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.016818,77.562598
8,2025-10-17 00:00:00.012,NaN,NaN,NaN,NaN,NaN,5.357397,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.016818,77.562598
9,2025-10-17 00:00:00.012,NaN,NaN,NaN,NaN,NaN,5.357399,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.016818,77.562598


In [33]:
# ✅ Step: Save pivoted dataset into the "data" folder
output_path = "data/pivoted_data.csv"
pivot_df.to_csv(output_path, index=False)

print(f"✅ Pivoted dataset saved successfully at: {output_path}")


✅ Pivoted dataset saved successfully at: data/pivoted_data.csv


In [34]:
# Get sorted unique PIDs
all_pids = sorted(df['PID'].unique().tolist())

# Display them horizontally (landscape)
print("Total number of unique PIDs:", len(all_pids))
print("\nList of PIDs:\n")
print(" | ".join(all_pids))


Total number of unique PIDs: 31

List of PIDs:

Altitude (GPS) | Average fuel consumption | Average fuel consumption (total) | Average fuel consumption 10 sec | Average speed | Average speed (GPS) | Barometric pressure | Calculated boost | Calculated engine load value | Calculated instant fuel consumption | Calculated instant fuel rate | Distance traveled with MIL on | Distance travelled | Distance travelled (total) | Engine RPM | Engine RPM x1000 | Engine coolant temperature | Fuel rail press. | Fuel used | Fuel used (total) | Fuel used price | Fuel used price (total) | Instant engine power (based on fuel consumption) | Intake air temperature | Intake manifold absolute pressure | MAF air flow rate | OBD Module Voltage | Power from MAF | Speed (GPS) | Vehicle acceleration | Vehicle speed


In [35]:
!pip install matplotlib


In [37]:
import pandas as pd

# Load the saved pivot (if not already in memory)
pivot_df = pd.read_csv("data/pivoted_data.csv")

# ✅ The 14 important PIDs
important_pids = [
    'Engine RPM',
    'Calculated engine load value',
    'Intake manifold absolute pressure',
    'MAF air flow rate',
    'Calculated instant fuel rate',
    'Engine coolant temperature',
    'Speed (GPS)',
    'Fuel rail pressure',
    'Power from MAF',
    'OBD Module Voltage',
    'Barometric pressure',
    'Ambient air temperature',
    'Intake air temperature',
    'Throttle position (manifold)'
]

# ✅ Ensure every required column exists (create empty if missing)
required_cols = ['timestamp'] + important_pids + ['LATITUDE', 'LONGITUDE']
for col in required_cols:
    if col not in pivot_df.columns:
        pivot_df[col] = pd.NA  # create empty column if missing

# ✅ Build the exact 17-column table in the required order
important_pids_df = pivot_df[required_cols]

print("✅ Important PIDs dataset created.")
print("Shape:", important_pids_df.shape)  # should be (6500, 17)
display(important_pids_df.head(10))


✅ Important PIDs dataset created.
Shape: (229877, 17)


,timestamp,Engine RPM,Calculated engine load value,Intake manifold absolute pressure,MAF air flow rate,Calculated instant fuel rate,Engine coolant temperature,Speed (GPS),Fuel rail pressure,Power from MAF,OBD Module Voltage,Barometric pressure,Ambient air temperature,Intake air temperature,Throttle position (manifold),LATITUDE,LONGITUDE
0,2025-10-17 00:00:00.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
1,2025-10-17 00:00:00.000,NaN,NaN,NaN,NaN,NaN,NaN,5.657209,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
2,2025-10-17 00:00:00.006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
3,2025-10-17 00:00:00.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
4,2025-10-17 00:00:00.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
5,2025-10-17 00:00:00.010,NaN,NaN,NaN,NaN,NaN,NaN,5.657209,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
6,2025-10-17 00:00:00.012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
7,2025-10-17 00:00:00.012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
8,2025-10-17 00:00:00.012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598
9,2025-10-17 00:00:00.012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,13.016818,77.562598


In [25]:
# ✅ Step: Save the corrected Important PIDs dataset (overwrite existing one)
output_path = "data/important_pids_dataset.csv"
important_pids_df.to_csv(output_path, index=False)

print(f"✅ Important PIDs dataset (17 columns) saved successfully at: {output_path}")


✅ Important PIDs dataset (17 columns) saved successfully at: data/important_pids_dataset.csv


In [38]:
import numpy as np

# ✅ Step 1: Compute overall sampling interval
# Convert timestamps to pandas datetime (if not already)
important_pids_df['timestamp'] = pd.to_datetime(important_pids_df['timestamp'])

# Compute differences between consecutive timestamps (in seconds)
time_diffs = important_pids_df['timestamp'].diff().dt.total_seconds()

# Drop NaNs and compute summary statistics
mean_interval = time_diffs.mean()
median_interval = time_diffs.median()
min_interval = time_diffs.min()
max_interval = time_diffs.max()

print("✅ Overall Sampling Interval (seconds):")
print(f"Mean   : {mean_interval:.4f}")
print(f"Median : {median_interval:.4f}")
print(f"Min    : {min_interval:.4f}")
print(f"Max    : {max_interval:.4f}")

print(f"\nApproximate overall sampling rate ≈ {1/mean_interval:.2f} Hz")


✅ Overall Sampling Interval (seconds):
Mean   : 0.0079
Median : 0.0000
Min    : 0.0000
Max    : 2.7860

Approximate overall sampling rate ≈ 125.97 Hz


/tmp/ipykernel_3817360/793494517.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  important_pids_df['timestamp'] = pd.to_datetime(important_pids_df['timestamp'])


In [ ]:
# | Metric                       | Meaning                              | Interpretation                                                                                                                                                    |
# | :--------------------------- | :----------------------------------- | :---------------------------------------------------------------------------------------------------------------------------------------------------------------- |
# | **Mean interval = 0.0079 s** | Average gap between consecutive rows | ≈ every **54 milliseconds**, i.e. **~18.4 samples per second** overall.                                                                                           |
# | **Median = 0.0000 s**        | Typical (most frequent) gap          | Many consecutive rows have **identical timestamps** — meaning multiple PIDs were recorded at the *same moment* (one sample containing different sensor readings). |
# | **Max = 2.7860 s**           | Longest observed gap                 | Occasionally, the logger didn’t record any update for ~9 seconds — likely due to PID switching, ECU delays, or no new data.                                       |


In [40]:
# ✅ Step 2: Compute sampling rate (Hz) for each PID separately
pid_sampling_summary = []

# Ensure timestamp is in datetime format
important_pids_df['timestamp'] = pd.to_datetime(important_pids_df['timestamp'])

# List of all PID columns (excluding timestamp, latitude, longitude)
pid_cols = [col for col in important_pids_df.columns if col not in ['timestamp', 'LATITUDE', 'LONGITUDE']]

for pid in pid_cols:
    # Keep only rows where this PID has a value
    temp = important_pids_df[['timestamp', pid]].dropna(subset=[pid])
    
    if len(temp) > 1:
        # Compute time difference between consecutive valid samples (in seconds)
        diffs = temp['timestamp'].diff().dt.total_seconds().dropna()
        mean_interval = diffs.mean()
        median_interval = diffs.median()
        
        # Compute approximate sampling rate (Hz)
        mean_rate = 1 / mean_interval if mean_interval and mean_interval > 0 else np.nan
        
        pid_sampling_summary.append({
            'PID': pid,
            'Mean_Interval_s': round(mean_interval, 4),
            'Median_Interval_s': round(median_interval, 4),
            'Approx_Sampling_Rate_Hz': round(mean_rate, 2),
            'Sample_Count': len(temp)
        })

# Convert to a DataFrame for easy viewing
pid_sampling_df = pd.DataFrame(pid_sampling_summary).sort_values(by='Approx_Sampling_Rate_Hz', ascending=False)

print("✅ PID-wise Sampling Rate Summary:")
display(pid_sampling_df)


✅ PID-wise Sampling Rate Summary:


/tmp/ipykernel_3817360/3088954650.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  important_pids_df['timestamp'] = pd.to_datetime(important_pids_df['timestamp'])


,PID,Mean_Interval_s,Median_Interval_s,Approx_Sampling_Rate_Hz,Sample_Count
1,Calculated engine load value,0.2031,0.209,4.92,8969
9,Barometric pressure,0.2084,0.212,4.80,8581
0,Engine RPM,0.2090,0.212,4.79,8558
2,Intake manifold absolute pressure,0.2090,0.212,4.79,8558
10,Intake air temperature,0.2090,0.212,4.79,8558
3,MAF air flow rate,0.2091,0.212,4.78,8552
4,Calculated instant fuel rate,0.2091,0.212,4.78,8552
8,OBD Module Voltage,0.2092,0.212,4.78,8548
7,Power from MAF,0.2091,0.212,4.78,8552
5,Engine coolant temperature,0.3877,0.407,2.58,4699


In [42]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("data/important_pids_dataset.csv")

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Separate numeric and non-numeric columns (excluding timestamp)
numeric_cols = []
non_numeric_cols = []

for col in df.columns:
    if col != 'timestamp':
        # Try to convert to numeric silently
        temp = pd.to_numeric(df[col], errors='coerce')
        # If at least half of the column is numeric, treat as numeric
        if temp.notna().sum() >= len(df) / 2:
            df[col] = temp
            numeric_cols.append(col)
        else:
            non_numeric_cols.append(col)

# Group by timestamp
grouped = df.groupby('timestamp')

# Compute mean for numeric cols and first non-null for non-numeric cols
numeric_df = grouped[numeric_cols].mean().reset_index()
non_numeric_df = grouped[non_numeric_cols].first().reset_index()

# Merge both parts back together
df_unique = pd.merge(numeric_df, non_numeric_df, on='timestamp', how='outer')

# Sort by timestamp (optional)
df_unique = df_unique.sort_values('timestamp').reset_index(drop=True)

print("✅ Unique timestamps processed successfully.")
print("Numeric columns averaged; non-numeric columns kept as first valid entry.")
print("Shape after grouping:", df_unique.shape)
display(df_unique.head(10))


✅ Unique timestamps processed successfully.
Numeric columns averaged; non-numeric columns kept as first valid entry.
Shape after grouping: (2292, 17)


,timestamp,LATITUDE,LONGITUDE,Engine RPM,Calculated engine load value,Intake manifold absolute pressure,MAF air flow rate,Calculated instant fuel rate,Engine coolant temperature,Speed (GPS),Fuel rail pressure,Power from MAF,OBD Module Voltage,Barometric pressure,Ambient air temperature,Intake air temperature,Throttle position (manifold)
0,2025-10-17 00:00:00.000,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-10-17 00:00:00.008,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-10-17 00:00:00.009,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-10-17 00:00:00.013,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-10-17 00:00:00.014,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2025-10-17 00:00:04.772,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-10-17 00:00:04.773,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2025-10-17 00:00:04.827,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2025-10-17 00:00:05.247,13.015922,77.565655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2025-10-17 00:00:05.248,13.015922,77.565655,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
import pandas as pd

# Assuming your cleaned dataframe is called df_unique

# Drop rows where all PIDs (except timestamp, LATITUDE, LONGITUDE) are NaN
pid_columns = [col for col in df_unique.columns if col not in ['timestamp', 'LATITUDE', 'LONGITUDE']]
df_non_nan = df_unique.dropna(subset=pid_columns, how='all')

# Reset index for clean output
df_non_nan = df_non_nan.reset_index(drop=True)

print("✅ Non-NaN rows extracted successfully.")
print("Original shape:", df_unique.shape)
print("After filtering:", df_non_nan.shape)
display(df_non_nan.head(10))


✅ Non-NaN rows extracted successfully.
Original shape: (2292, 17)
After filtering: (1465, 17)


,timestamp,LATITUDE,LONGITUDE,Engine RPM,Calculated engine load value,Intake manifold absolute pressure,MAF air flow rate,Calculated instant fuel rate,Engine coolant temperature,Speed (GPS),Fuel rail pressure,Power from MAF,OBD Module Voltage,Barometric pressure,Ambient air temperature,Intake air temperature,Throttle position (manifold)
0,2025-10-17 00:00:00.000,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-10-17 00:00:00.013,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-10-17 00:00:00.014,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-10-17 00:00:04.772,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-10-17 00:00:04.773,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2025-10-17 00:00:04.827,13.015782,77.565621,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-10-17 00:00:05.248,13.015922,77.565655,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2025-10-17 00:00:11.421,13.015813,77.565668,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2025-10-17 00:00:11.425,13.015813,77.565668,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2025-10-17 00:00:11.427,13.015813,77.565668,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
# ✅ Save the filtered dataset
output_path = "data/data_without_NaN.csv"
df_non_nan.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved successfully at: {output_path}")
print("File contains only timestamps with at least one valid PID reading.")


✅ Cleaned dataset saved successfully at: data/data_without_NaN.csv
File contains only timestamps with at least one valid PID reading.


In [45]:
valid_counts = df_non_nan.notna().sum().sort_values(ascending=False)
print(valid_counts)


timestamp                            1465
LATITUDE                             1465
LONGITUDE                            1465
Calculated engine load value          388
Speed (GPS)                           258
Engine RPM                            215
Engine coolant temperature            188
Calculated instant fuel rate          171
MAF air flow rate                     171
Power from MAF                        171
Intake manifold absolute pressure      90
OBD Module Voltage                     65
Barometric pressure                    54
Intake air temperature                 38
Fuel rail pressure                      0
Ambient air temperature                 0
Throttle position (manifold)            0
dtype: int64
